# Phase 3 — Build Evaluated Semantic Retrieval

**Roadmap reference:** `ContractIQ_Project_Execution_Roadmap.md`, Phase 3

**Goal:** clause-boundary-aware chunks, traced embeddings, a persistent ChromaDB index with idempotent upserts, metadata-filtered search, and a labeled retrieval evaluation (not spot-checks).

**Completion gate:** the persisted index reloads after restart; filtered retrieval respects scope; returned passages have resolvable provenance; retrieval quality is measured against labeled examples.

## 1. Load configuration and the manifest

In [17]:
import sys
sys.path.append("..")

from src.contractiq import config
from src.contractiq.ingestion import load_manifest
from src.contractiq.retrieval import (
    chunk_all_documents,
    compare_chunk_configs,
    build_index,
    verify_index_reload,
    search,
    search_hybrid,
    save_retrieval_test_set_template,
    load_retrieval_test_set,
    evaluate_retrieval,
    suggest_test_set_candidates
)

manifest = load_manifest()
print(f"Loaded manifest: {len(manifest)} documents")

ImportError: cannot import name 'search_hybrid' from 'src.contractiq.retrieval' (e:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\PWC_CONTRACTIQ_PROJECT\Research\..\src\contractiq\retrieval.py)

## 2. Compare chunk configurations before committing to one

The roadmap's own rule: start at 600-800 words / 80-120 overlap, then MEASURE rather than assume. This compares chunk-count and size distribution for a few candidates — pick one, then chunk the full corpus with it in step 3.

In [2]:
candidate_configs = [
    {"target_words": 600, "overlap_words": 80},
    {"target_words": 700, "overlap_words": 100},
    {"target_words": 800, "overlap_words": 120},
]

comparison = compare_chunk_configs(manifest, candidate_configs)

Chunked 36 documents into 72 chunks (target=600w, overlap=80w).
Chunked 36 documents into 63 chunks (target=700w, overlap=100w).
Chunked 36 documents into 56 chunks (target=800w, overlap=120w).
  target=600  overlap=80   ->   72 chunks, avg 452.9 words (min 86, max 600)
  target=700  overlap=100  ->   63 chunks, avg 519.8 words (min 142, max 700)
  target=800  overlap=120  ->   56 chunks, avg 583.4 words (min 124, max 800)


## 3. Chunk the full corpus with the chosen configuration

Defaults to `config.CHUNK_SIZE_WORDS` / `config.CHUNK_OVERLAP_WORDS` (700/100) — update those in `config.py` if step 2 pointed elsewhere.

In [3]:
chunks_by_doc = chunk_all_documents(manifest)

sample_doc_id = next(iter(chunks_by_doc))
sample_chunk = chunks_by_doc[sample_doc_id][0]
print(f"\nSample chunk from {sample_doc_id}:")
print(f"  chunk_id: {sample_chunk.chunk_id}")
print(f"  source_location_range: {sample_chunk.source_location_range}")
print(f"  text (first 200 chars): {sample_chunk.text[:200]}")

Chunked 36 documents into 63 chunks (target=700w, overlap=100w).

Sample chunk from COM-001:
  chunk_id: COM-001-c0
  source_location_range: paragraph:0-paragraph:46
  text (first 200 chars): Sample 1: Pricing Table Agreement
Between: Primevolta Hardware Solutions (Vendor)
And: FinSecure Services (Client)
Date: 11 December 2025
1. Introduction
This Pricing Table Agreement (“Agreement”) is 


## 4. Build the persistent ChromaDB index

Run this cell twice in a row — the second run should show 0 chunks needing re-embedding. That's the idempotency check.

In [4]:
collection = build_index(manifest, chunks_by_doc)

0 of 63 chunk(s) need (re)embedding; 63 unchanged and skipped.
Index now has 63 chunk(s) total.


In [5]:
# Idempotency check: re-running should embed 0 new chunks
collection = build_index(manifest, chunks_by_doc)

0 of 63 chunk(s) need (re)embedding; 63 unchanged and skipped.
Index now has 63 chunk(s) total.


## 5. Verify the index survives a restart

Completion-gate requirement: the persisted index reloads after restart.

In [6]:
verify_index_reload()

Reopened index from disk: 63 chunk(s) found.


63

## 6. Try filtered search

In [7]:
results = search("What happens if the vendor is late on delivery?", top_k=5)
for r in results:
    print(f"[{r['score']:.3f}] {r['doc_id']} ({r['category_key']}) "
          f"@ {r['source_location_range']}")
    print(f"    {r['text'][:150]}\n")

[0.451] MLA-001 (master_agreements) @ paragraph:12-paragraph:26
    Furthermore, the Client may temporarily suspend or terminate services with immediate effect if the Vendor’s activities pose imminent risk to security,

[0.435] MLA-002 (master_agreements) @ paragraph:15-paragraph:29
    5. SERVICE LEVEL COMMITMENTS AND PERFORMANCE EXPECTATIONS
The Vendor shall deliver all managed services in accordance with well-defined service level 

[0.409] MLA-001 (master_agreements) @ paragraph:24-paragraph:43
    The warranty does not extend to defects arising from misuse, unauthorized modifications, environmental damage, power fluctuations, exposure to liquids

[0.407] FIN-001 (financial_billing) @ paragraph:0-paragraph:44
    Primevolta Hardware Solutions
Billing Schedule Agreement
Client: FinSecure Services
Date: 11th December 2025
1. Purpose
This Billing Schedule Agreemen

[0.407] CMP-005 (compliance_docs) @ paragraph:0-paragraph:68
    Primevolta Hardware Solutions – Vendor Management Policy

In [8]:
# Same query, scoped to only master_agreements
results_scoped = search("What happens if the vendor is late on delivery?",
                        top_k=5, category_key="master_agreements")
for r in results_scoped:
    print(f"[{r['score']:.3f}] {r['doc_id']} @ {r['source_location_range']}")

[0.451] MLA-001 @ paragraph:12-paragraph:26
[0.435] MLA-002 @ paragraph:15-paragraph:29
[0.409] MLA-001 @ paragraph:24-paragraph:43
[0.405] MLA-005 @ paragraph:39-paragraph:47
[0.395] MLA-006 @ paragraph:15-paragraph:27


## 7. Build the retrieval test set — with candidate suggestions

`save_retrieval_test_set_template()` writes an empty file with 15 slots (3 each: liability, payment, termination, SLA, confidentiality).

`suggest_test_set_candidates()` then runs real semantic search for a set of draft questions (plain text — not evidence) and prints the top 3 results with a readable excerpt for each. Read the excerpt, decide if it genuinely answers the question, and copy that result's `doc_id` and `source_location_range` into the JSON file yourself. This is still a manual step — the tool narrows 36 documents down to 3 candidates per question, it does not decide correctness for you.

In [11]:
template_path = save_retrieval_test_set_template()
suggest_test_set_candidates(manifest)

Already exists, not overwriting: E:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\PWC_CONTRACTIQ_PROJECT\artifacts\evaluations\phase3_retrieval_test_set.json

LIABILITY

Q: Is there a cap on the vendor's total liability under this agreement?
  [0.517] MLA-001    (MSA Sample 1.docx) @ paragraph:24-paragraph:43
      The warranty does not extend to defects arising from misuse, unauthorized modifications, environmental damage, power fluctuations, exposure to liquids, or any handling that contravenes the usage guidelines issued by the Vendor. Firmware and software elements integral to the Devic...
  [0.515] MLA-001    (MSA Sample 1.docx) @ paragraph:40-paragraph:46
      11. GOVERNING LAW AND DISPUTE RESOLUTION This Agreement shall be governed by and interpreted in accordance with the laws of the jurisdiction specified in the SOW. Any dispute, controversy, or claim arising out of or relating to this Agree

---
### Manual step before continuing

Open `artifacts/evaluations/phase3_retrieval_test_set.json`, and for each of the 15 rows fill in:
- `question` — a real question a reviewer would ask
- `expected_doc_id` — the doc_id that actually answers it
- `expected_locator_contains` — a substring like `"paragraph:14"` you manually confirmed contains the answer

Then run the cell below.

## 8. Evaluate retrieval against the labeled test set

In [16]:
test_set = load_retrieval_test_set()
eval_report = evaluate_retrieval(test_set, top_k=5)

Recall@5: 100.0% (14/14)
  ✓ [liability] Is there a cap on the vendor's total liability under this ag
  ✓ [liability] If the vendor is negligent and damages the client's property
  ✓ [liability] Are consequential or indirect damages excluded under this co
  ✓ [payment] What is the payment due date or payment term under this agre
  ✓ [payment] What happens if an invoice is not paid on time - is there a 
  ✓ [payment] What is the total contract value or pricing structure descri
  ✓ [termination] Under what conditions can either party terminate this agreem
  ✓ [termination] What notice period is required to terminate the agreement wi
  ✓ [termination] What happens to outstanding obligations or fees upon termina
  ✓ [sla] What service level or uptime commitment does the vendor make
  ✓ [sla] What remedy or credit applies if the vendor misses a service
  ✓ [sla] How is service performance measured or reported under this a
  ✓ [confidentiality] How long do confidentiality obligations survive

In [14]:
test_set = load_retrieval_test_set()
eval_report = evaluate_retrieval(test_set, top_k=3)

Recall@3: 100.0% (14/14)
  ✓ [liability] Is there a cap on the vendor's total liability under this ag
  ✓ [liability] If the vendor is negligent and damages the client's property
  ✓ [liability] Are consequential or indirect damages excluded under this co
  ✓ [payment] What is the payment due date or payment term under this agre
  ✓ [payment] What happens if an invoice is not paid on time - is there a 
  ✓ [payment] What is the total contract value or pricing structure descri
  ✓ [termination] Under what conditions can either party terminate this agreem
  ✓ [termination] What notice period is required to terminate the agreement wi
  ✓ [termination] What happens to outstanding obligations or fees upon termina
  ✓ [sla] What service level or uptime commitment does the vendor make
  ✓ [sla] What remedy or credit applies if the vendor misses a service
  ✓ [sla] How is service performance measured or reported under this a
  ✓ [confidentiality] How long do confidentiality obligations survive

## 9. Hybrid search experiment (dense + BM25 via Reciprocal Rank Fusion)

Dense embedding search alone already hit 100% recall@5 on the labeled test
set above. Before assuming hybrid retrieval would help anyway (a common
default recommendation), it's measured here directly on the same 14
questions: `search_hybrid()` runs BM25 lexical scoring over the same chunks
and fuses it with the dense ranking via RRF (rank-based fusion, so cosine
similarity and BM25's unbounded score never need to be normalized against
each other).


In [19]:
test_set = load_retrieval_test_set()

print("--- DENSE (baseline) ---")
dense_report = evaluate_retrieval(test_set, top_k=5, search_fn=search, report_suffix="")

print("\n--- HYBRID (dense + BM25, RRF) ---")
hybrid_report = evaluate_retrieval(test_set, top_k=5, search_fn=search_hybrid, report_suffix="_hybrid")

print(f"\nDense recall@5:  {dense_report['recall_at_k']:.1%}")
print(f"Hybrid recall@5: {hybrid_report['recall_at_k']:.1%}")


--- DENSE (baseline) ---
Recall@5: 100.0% (14/14)
  ✓ [liability] Is there a cap on the vendor's total liability under this ag
  ✓ [liability] If the vendor is negligent and damages the client's property
  ✓ [liability] Are consequential or indirect damages excluded under this co
  ✓ [payment] What is the payment due date or payment term under this agre
  ✓ [payment] What happens if an invoice is not paid on time - is there a 
  ✓ [payment] What is the total contract value or pricing structure descri
  ✓ [termination] Under what conditions can either party terminate this agreem
  ✓ [termination] What notice period is required to terminate the agreement wi
  ✓ [termination] What happens to outstanding obligations or fees upon termina
  ✓ [sla] What service level or uptime commitment does the vendor make
  ✓ [sla] What remedy or credit applies if the vendor misses a service
  ✓ [sla] How is service performance measured or reported under this a
  ✓ [confidentiality] How long do confidenti

### Verdict: hybrid rejected, dense-only kept

Hybrid **regressed 2 previously-correct queries** instead of fixing anything:
dense-only was already at 100% recall@5, so there was nothing left to gain,
and BM25's exact term-frequency matching pulled in lexically-similar but
topically wrong documents — e.g. `"outstanding obligations upon termination"`
(expects `MLA-002`) lost its correct hit because BM25 matched the literal
word **"outstanding"** against billing/invoice documents ("outstanding
balance") that have nothing to do with contract termination.

`search_hybrid()` stays in `retrieval.py` as a documented, measured negative
result — not wired into Phase 4's agent, which still uses plain `search()`.
</VSCode.Cell>


---
### Next: `04_Legal_Risk_Agent.ipynb`

Build the first evidence-grounded specialist agent — retrieve relevant clause evidence with this search function, then require every legal finding to cite real evidence IDs and excerpts.